In [3]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
import gc
import hydra
import logging
from omegaconf import DictConfig, OmegaConf
import torch
import wandb
torch.cuda.empty_cache()

# from analysis_utils import analyse_repertoire
from diffusers import (
    UNet2DConditionModel,
    AutoencoderKL,
    LMSDiscreteScheduler
)
from transformers import (
    AutoImageProcessor,
    AutoModel,
    CLIPModel,
    CLIPTextModel,
    CLIPTokenizerFast,
)
from feature_functions import (
    brightness,
    average_hue,
    get_artclip_score_model,
    dino_feature_extractor
)

# from model_utils import (
#     expand_prompt_embedding,
#     toggle_prompt_embedding_shape,
#     get_text_embeddings,
#     generate_images_from_latents,
#     fitness_function,
#     save_images
# )

import jax
import jax.numpy as jnp
import numpy as np
from functools import partial
from qdax.baselines.cmaes import CMAES
from qdax.core.emitters.mutation_operators import isoline_variation, slerp_variation
from qdax.core.emitters.standard_emitters import MixingEmitter
from qdax.core.containers.mapelites_repertoire import compute_cvt_centroids, MapElitesRepertoire
from qdax.core.map_elites import MAPElites
from qdax.utils.metrics import default_qd_metrics, CSVLogger
from qdax.utils.plotting import plot_map_elites_results
from typing import Callable, List
import time
from tqdm import tqdm


In [4]:
# Function to encode text prompt into embeddings
def get_text_embeddings(prompt,
        batch_size,
        tokenizer,
        text_encoder,
        with_grad=False
    ):
    text_input = tokenizer([str(prompt)] * batch_size, padding="max_length", max_length=77, return_tensors="pt")

    if with_grad:
        # Enable gradients for text encoding
        text_embeddings = text_encoder(text_input.input_ids.cuda())[0]
    else:
        with torch.no_grad():
            text_embeddings = text_encoder(text_input.input_ids.cuda())[0]
    
    # Generate unconditional embeddings for classifier-free guidance
    uncond_input = tokenizer([""] * batch_size, padding="max_length", max_length=77, return_tensors="pt")
    
    if with_grad:
        uncond_embeddings = text_encoder(uncond_input.input_ids.cuda())[0]
    else:
        with torch.no_grad():
            uncond_embeddings = text_encoder(uncond_input.input_ids.cuda())[0]
            
    print(f"Text embeddings shape: {text_embeddings.shape}, Uncond embeddings shape: {uncond_embeddings.shape}")
    
    # Concatenate conditional & unconditional embeddings
    text_embeddings = torch.cat([uncond_embeddings, text_embeddings]) 
    return text_embeddings.detach()

In [6]:
tokenizer = CLIPTokenizerFast.from_pretrained("stabilityai/sd-turbo", subfolder="tokenizer", torch_dtype=torch.float32)
text_encoder = CLIPTextModel.from_pretrained("stabilityai/sd-turbo", subfolder="text_encoder", torch_dtype=torch.float32).cuda().eval()


embeds = get_text_embeddings(
    prompt="cat",
    batch_size=16,
    tokenizer=tokenizer,
    text_encoder=text_encoder,
    with_grad=False,
)

Text embeddings shape: torch.Size([16, 77, 1024]), Uncond embeddings shape: torch.Size([16, 77, 1024])


In [7]:
embeds.shape

torch.Size([32, 77, 1024])

In [8]:
single_embeds = get_text_embeddings(
    prompt="cat",
    batch_size=1,
    tokenizer=tokenizer,
    text_encoder=text_encoder,
    with_grad=False,
)

Text embeddings shape: torch.Size([1, 77, 1024]), Uncond embeddings shape: torch.Size([1, 77, 1024])


In [9]:
single_embeds.shape

torch.Size([2, 77, 1024])

In [15]:
unconds = np.full((16, 77, 1024), True, dtype=bool)
conds = np.full((16, 77, 1024), False, dtype=bool)

In [52]:
concat = torch.cat([torch.tensor(unconds), torch.tensor(conds)], dim=0)
concat.shape

torch.Size([32, 77, 1024])

In [28]:
single_embed = torch.cat([torch.tensor(np.full((1, 77, 1024), True)), torch.tensor(np.full((1, 77, 1024), False))])

In [29]:
single_embed.shape

torch.Size([2, 77, 1024])

In [30]:
single_embed

tensor([[[ True,  True,  True,  ...,  True,  True,  True],
         [ True,  True,  True,  ...,  True,  True,  True],
         [ True,  True,  True,  ...,  True,  True,  True],
         ...,
         [ True,  True,  True,  ...,  True,  True,  True],
         [ True,  True,  True,  ...,  True,  True,  True],
         [ True,  True,  True,  ...,  True,  True,  True]],

        [[False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         ...,
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False]]])

In [31]:
def expand_prompt_embedding(prompt_embedding, batch_size):
    """
    Takes a [2, 77, 1024] prompt embedding (uncond, cond) and returns
    [2*batch_size, 77, 1024] with batch_size uncond followed by batch_size cond.
    """
    uncond = prompt_embedding[0].unsqueeze(0).repeat(batch_size, 1, 1)  # [batch_size, 77, 1024]
    cond = prompt_embedding[1].unsqueeze(0).repeat(batch_size, 1, 1)    # [batch_size, 77, 1024]
    expanded = torch.cat([uncond, cond], dim=0)  # [2 * batch_size, 77, 1024]
    return expanded

In [35]:
expanded = expand_prompt_embedding(single_embed, batch_size=16)

In [36]:
def toggle_prompt_embedding_shape(embedding):
    """
    Toggle shape between:
    - [2*B, 77, 1024] → [B, 2, 77, 1024]
    - [B, 2, 77, 1024] → [2*B, 77, 1024]

    Works with both PyTorch tensors and JAX arrays.
    Assumes uncond + cond are stacked vertically in the 2B form.
    """
    # Determine backend
    if isinstance(embedding, torch.Tensor):
        backend = "torch"
        ndim = embedding.ndim
    elif isinstance(embedding, jnp.ndarray):
        backend = "jax"
        ndim = embedding.ndim
    else:
        raise TypeError(f"Unsupported type: {type(embedding)}")

    # Toggle shapes
    if ndim == 3:
        B = embedding.shape[0] // 2
        uncond = embedding[:B]
        cond = embedding[B:]
        if backend == "torch":
            return torch.stack([uncond, cond], dim=1)
        else:
            return jnp.stack([uncond, cond], axis=1)

    elif ndim == 4:
        uncond = embedding[:, 0]
        cond = embedding[:, 1]
        if backend == "torch":
            return torch.cat([uncond, cond], dim=0)
        else:
            return jnp.concatenate([uncond, cond], axis=0)

    else:
        raise ValueError(f"Expected input with 3 or 4 dimensions, got {ndim}")

In [50]:
toggled = toggle_prompt_embedding_shape(expanded)
toggled.shape

torch.Size([16, 2, 77, 1024])

In [51]:
toggle_prompt_embedding_shape(toggled).shape

torch.Size([32, 77, 1024])